# FakeGang GRPO — Cross-Platform Colab Notebook

End-to-end re-runnable **cross-platform** GRPO training for the fake-gang content-moderation policy, using **TRL's `GRPOTrainer`** (with optional **Unsloth** acceleration).

**What this does**
1. Installs deps (Unsloth + TRL + transformers).
2. Clones the training repo from HF (contains `training/`, prompts, reward fn, rollouts).
3. Points the training script at our remote env Space (no local server needed).
4. Runs **phase3** — joint training on **Instagram + X + Snapchat** (26 steps), then a zero-shot held-out eval on **LinkedIn**.
5. Plots `train/reward` (the cross-platform learning curve) plus supporting metrics.

**Recommended Colab runtime**: GPU → T4 (free) is enough for the 0.5B model. A100 if you want headroom.

---

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

Unsloth gives a ~2x training speedup and lower VRAM. If you prefer pure TRL (no Unsloth), skip the first line.

In [ ]:
# Unsloth (optional but recommended)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Pinned to match the HF Space training image (see training/requirements.txt).
!pip install -q --upgrade \
    "trl>=0.13.0" "transformers>=4.46.0" "datasets>=2.20.0" \
    "accelerate>=1.0.0" "peft>=0.13.0" "bitsandbytes>=0.45.0" \
    "huggingface_hub>=0.25.0" wandb requests tenacity

## 3. Clone the training repo

All the helper modules (`prompts.py`, `rollout.py`, `grounded_reward.py`, `build_dataset.py`) live in the HF Space. We clone it so `python -m training.train_grpo` works.

In [ ]:
import os
REPO_ID = "pandago/fakegang-grpo"  # HF Space holding the training code

if not os.path.isdir("/content/fakegang"):
    !git clone https://huggingface.co/spaces/{REPO_ID} /content/fakegang
else:
    !cd /content/fakegang && git pull --rebase

%cd /content/fakegang
!ls training/

## 4. Configure env URL + secrets

The training loop hits a remote env Space (FastAPI server) for episode resets, observations, and grader scores. **No local env required.**

Optional: set a W&B key to log live metrics.

In [ ]:
import os
from getpass import getpass

# Remote env Space — leave as-is unless you've deployed your own copy.
os.environ["ENV_BASE_URL"] = "https://pandago-graphstrike-model-training.hf.space"

# Optional: W&B login. Skip the cell if you don't want logging.
wandb_key = getpass("WANDB_API_KEY (blank to skip): ").strip()
if wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key
    os.environ["WANDB_PROJECT"] = "fakegang-grpo"

# Smoke-check the env
import requests
r = requests.get(os.environ["ENV_BASE_URL"] + "/health", timeout=15)
print("env /health:", r.status_code, r.text[:200])

## 5. (Optional) Patch the trainer to use Unsloth

Skip this cell if you want pure TRL. With Unsloth enabled, the same `GRPOTrainer` runs ~2x faster — Qwen2.5-0.5B-Instruct fits a free-tier T4 with comfortable headroom either way.

In [ ]:
USE_UNSLOTH = True

if USE_UNSLOTH:
    from unsloth import FastLanguageModel, PatchFastRL
    # Patches TRL's GRPOTrainer to use Unsloth's fused kernels for generation + backward.
    PatchFastRL("GRPO", FastLanguageModel)
    print("Unsloth GRPO patch applied.")
else:
    print("Running pure TRL (no Unsloth).")

## 6. Run training — phase3 (cross-platform demo, 26 steps)

This is the **cross-platform** run: trains jointly on **Instagram + X + Snapchat**, then automatically runs a zero-shot **held-out evaluation on LinkedIn** to test generalization.

Phase reference (see `training/train_grpo.py`):
- `phase0` — baseline rollouts only, no training
- `phase1` — smoke (10 steps, single platform) — verifies gradients flow
- `phase2` — signal (25 steps, single platform)
- `phase3` — **cross-platform demo (26 steps × 4 gens × 3 platforms → LinkedIn held-out eval)** ← *this run*

In [ ]:
PHASE          = "phase3"
MODEL          = "Qwen/Qwen2.5-0.5B-Instruct"
TRAIN_PLATFORMS = ["Instagram", "X", "Snapchat"]
EVAL_PLATFORM  = "LinkedIn"
OUT_DIR        = "/content/runs"
WANDB_PROJ     = os.environ.get("WANDB_PROJECT", "")

extra = f"--wandb-project {WANDB_PROJ}" if WANDB_PROJ else ""

!python -m training.train_grpo \
    --phase {PHASE} \
    --model {MODEL} \
    --platforms {' '.join(TRAIN_PLATFORMS)} \
    --eval-platform {EVAL_PLATFORM} \
    --base-url {os.environ['ENV_BASE_URL']} \
    --out-dir {OUT_DIR} \
    {extra}

## 7. Plot training curves

The hero plot is **`train/reward`** — the cross-platform learning curve across Instagram + X + Snapchat over 26 steps. Expected: clear upward trend (~0.1 → ~0.30).

Supporting plots cover loss, KL divergence, reward std (gradient signal health), and completion length (collapse check). The held-out **LinkedIn** eval result (run automatically after training) is reported separately in the script output.

In [ ]:
import json, glob
import numpy as np
import matplotlib.pyplot as plt

log_paths = sorted(glob.glob(f"{OUT_DIR}/{PHASE}/**/trainer_state.json", recursive=True))
assert log_paths, "no trainer_state.json found — did training finish?"
state = json.load(open(log_paths[-1]))
hist  = state["log_history"]

def series(key):
    xs, ys = [], []
    for h in hist:
        if key in h and "step" in h:
            xs.append(h["step"]); ys.append(h[key])
    return xs, ys

# ── Hero plot: train/reward ──────────────────────────────────────────────────
sx, sy = series("reward")
plt.figure(figsize=(10, 5))
plt.plot(sx, sy, marker="o", linewidth=2, color="#2E86AB", label="train/reward")
if len(sy) >= 5:
    win = max(3, len(sy) // 5)
    smooth = np.convolve(sy, np.ones(win)/win, mode="valid")
    plt.plot(sx[win-1:], smooth, linewidth=2.5, color="#E63946",
             label=f"smoothed (window={win})")
plt.title(f"train/reward — {PHASE} ({MODEL.split('/')[-1]})", fontsize=14, fontweight="bold")
plt.xlabel("step"); plt.ylabel("mean reward")
plt.grid(alpha=0.3); plt.legend(loc="lower right")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/{PHASE}/train_reward.png", dpi=140)
plt.show()

if sy:
    print(f"train/reward  first={sy[0]:.3f}  last={sy[-1]:.3f}  "
          f"delta={sy[-1]-sy[0]:+.3f}  ({(sy[-1]/max(sy[0],1e-6)):.1f}× improvement)")

# ── Supporting metrics grid ──────────────────────────────────────────────────
panels = [
    ("loss",                "train/loss",            "#264653"),
    ("kl",                  "train/kl",              "#F4A261"),
    ("reward_std",          "train/reward_std",      "#2A9D8F"),
    ("completion_length",   "train/completion_len",  "#9B5DE5"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (key, label, color) in zip(axes.flat, panels):
    xs, ys = series(key)
    if not ys:
        ax.text(0.5, 0.5, f"no `{key}` logged", ha="center", va="center",
                transform=ax.transAxes, color="gray")
        ax.set_title(label); continue
    ax.plot(xs, ys, marker=".", linewidth=1.4, color=color)
    ax.set_title(label); ax.set_xlabel("step"); ax.grid(alpha=0.3)

fig.suptitle(f"Supporting metrics — {PHASE}", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/{PHASE}/train_metrics.png", dpi=140, bbox_inches="tight")
plt.show()

print(f"\nSaved plots:\n  {OUT_DIR}/{PHASE}/train_reward.png\n  {OUT_DIR}/{PHASE}/train_metrics.png")

## 8. (Optional) Push the trained checkpoint to HF Hub

In [ ]:
PUSH = False  # flip to True to push
PUSH_REPO_ID = "your-username/fakegang-grpo-phase2"

if PUSH:
    from huggingface_hub import login, HfApi
    login(getpass("HF token (write): ").strip())
    HfApi().upload_folder(
        folder_path=f"{OUT_DIR}/{PHASE}",
        repo_id=PUSH_REPO_ID,
        repo_type="model",
        commit_message=f"checkpoint: {PHASE}",
    )
    print(f"pushed → https://huggingface.co/{PUSH_REPO_ID}")

---
## Reproducibility notes for judges
- The remote env Space (`pandago-graphstrike-model-training.hf.space`) is open and rate-limited; restart from cell 1 if `/health` fails.
- **Cross-platform demo run**: phase3, **26 GRPO steps × 4 generations/step** on **Qwen2.5-0.5B-Instruct**, training jointly on **Instagram + X + Snapchat**.
- **Held-out generalization**: after training, the policy is zero-shot evaluated on **LinkedIn** (never seen during training). Eval metrics print at the end of the run and log to W&B as `eval/LinkedIn/*`.
- **Key plot — `train/reward`**: cross-platform learning curve, ~0.1 → ~0.30 across 26 steps.
- Supporting plots (`loss`, `kl`, `reward_std`, `completion_length`) verify training health.
- All hyperparameters live in `PHASE_CONFIG` at the top of `training/train_grpo.py`.
- Saved artifacts: `runs/phase3/train_reward.png`, `runs/phase3/train_metrics.png`, `runs/eval_linkedin.jsonl`, `trainer_state.json`, model checkpoint.